# Bài 1: Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột.

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession
    .builder
    .appName("Read file CSV")
    .master("local[*]")             # Sử dụng tất cả các core của máy local
    .getOrCreate()                  # Tạo hoặc lấy một SparkSession hiện có
)

26/05/20 13:27:22 WARN Utils: Your hostname, xuanhoang-Yoga-Slim-7-Pro-14ACH5-O resolves to a loopback address: 127.0.1.1; using 10.45.113.243 instead (on interface wlp1s0)
26/05/20 13:27:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/20 13:27:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [15]:
def print_schema(link_url):
    print(f"Schema of {link_url}")
    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .option("delimiter", ";") \
        .csv(f"data/{link_url}")
    df.printSchema()
    return df 

In [16]:
# Doc tat ca cac file trong folder data
file_name = ['Customer_List.csv', 'Order_Items.csv', 'Order_Reviews.csv', 'Orders.csv', 'Products.csv']
for file in file_name:
    df = print_schema(file)


Schema of Customer_List.csv
root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: timestamp (nullable = true)
 |-- First_Order_Date: timestamp (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)

Schema of Order_Items.csv
root
 |-- Order_ID: string (nullable = true)
 |-- Order_Item_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Seller_ID: string (nullable = true)
 |-- Shipping_Limit_Date: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Freight_Value: double (nullable = true)

Schema of Order_Reviews.csv
root
 |-- Review_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Review_Score: string (nullable = true)
 |-- Re

# Bài 2: Thống kê tổng số đơn hàng, số lượng khách hàng và người bán.


In [19]:
# Tổng số đơn hàng 
df_orders = print_schema("Orders.csv")
total_orders = df_orders.select("Order_ID").distinct().count()

print(f"Total Orders = {total_orders}")

# Tổng số khách hàng 
df_customers = print_schema("Customer_List.csv")
total_customers = df_customers.select("Customer_Trx_ID").distinct().count()

print(f"Total Customers = {total_customers}")

# Tổng số người bán 
df_order_items = print_schema("Order_Items.csv")
total_sellers = df_order_items.select("Seller_ID").distinct().count()

print(f"Total Sellers = {total_sellers}")

Schema of Orders.csv
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)

Total Orders = 99441
Schema of Customer_List.csv
root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: timestamp (nullable = true)
 |-- First_Order_Date: timestamp (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)

Total Customers = 994

# Bài 3: Phân tích số lượng đơn hàng theo từng quốc gia, sắp xếp theo thứ tự giảm dần 

In [20]:
from pyspark.sql import functions as F

# 1. Join Orders với Customers
df = df_orders.join(
    df_customers,
    on="Customer_Trx_ID",
    how="left"
)

# 2. Đếm số đơn hàng theo quốc gia
orders_by_country = df.groupBy("Customer_Country") \
    .agg(F.countDistinct("Order_ID").alias("total_orders")) \
    .orderBy(F.desc("total_orders"))

orders_by_country.show()

+----------------+------------+
|Customer_Country|total_orders|
+----------------+------------+
|         Germany|       41754|
|          France|       12848|
|     Netherlands|       11629|
|         Belgium|        5464|
|         Austria|        5043|
|     Switzerland|        3640|
|  United Kingdom|        3382|
|          Poland|        2139|
|         Czechia|        2034|
|           Italy|        2025|
|           Spain|        1651|
|        Portugal|        1336|
|          Sweden|         975|
|         Denmark|         905|
|          Serbia|         746|
|          Norway|         716|
|        Slovakia|         534|
|        Slovenia|         495|
|          Turkey|         485|
|          Greece|         412|
+----------------+------------+
only showing top 20 rows



# Bài 4: Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm tăng dần, tháng giảm dần)

In [21]:
from pyspark.sql import functions as F

df = df_orders

# 1. Extract year & month
df_time = df.withColumn("year", F.year("Order_Purchase_Timestamp")) \
            .withColumn("month", F.month("Order_Purchase_Timestamp"))

# 2. Group + count orders
orders_by_time = df_time.groupBy("year", "month") \
    .agg(F.countDistinct("Order_ID").alias("total_orders"))

# 3. Sort: year ASC, month DESC
result = orders_by_time.orderBy(F.asc("year"), F.desc("month"))

result.show()

+----+-----+------------+
|year|month|total_orders|
+----+-----+------------+
|2022|   12|           1|
|2022|   10|         324|
|2022|    9|           4|
|2023|   12|        5673|
|2023|   11|        7544|
|2023|   10|        4631|
|2023|    9|        4285|
|2023|    8|        4331|
|2023|    7|        4026|
|2023|    6|        3245|
|2023|    5|        3700|
|2023|    4|        2404|
|2023|    3|        2682|
|2023|    2|        1780|
|2023|    1|         800|
|2024|   10|           4|
|2024|    9|          16|
|2024|    8|        6512|
|2024|    7|        6292|
|2024|    6|        6167|
+----+-----+------------+
only showing top 20 rows



# Bài 5: Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến 5).
Lưu ý: Cần xử lý các giá trị ngoại lệ và NULL trong cột Review_Score

In [ ]:
df = print_schema('Order_Reviews.csv')

# Chuẩn hoá:
# ép về integer
# lọc NULL
# chỉ giữ 1–5

df_clean = df.withColumn(
    "Review_Score_Int",
    F.col("Review_Score").cast("int")
).filter(
    F.col("Review_Score_Int").isNotNull()
).filter(
    (F.col("Review_Score_Int") >= 1) &
    (F.col("Review_Score_Int") <= 5)
)

# Thống kê trung bình + số lượng
result = df_clean.groupBy("Review_Score_Int") \
    .agg(
        F.count("*").alias("total_reviews"),
        F.avg("Review_Score_Int").alias("avg_score")
    ) \
    .orderBy("Review_Score_Int")

result.show()



Schema of Order_Reviews.csv
root
 |-- Review_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Review_Score: string (nullable = true)
 |-- Review_Comment_Title_En: string (nullable = true)
 |-- Review_Comment_Message_En: string (nullable = true)
 |-- Review_Creation_Date: string (nullable = true)
 |-- Review_Answer_Timestamp: timestamp (nullable = true)

+----------------+-------------+---------+
|Review_Score_Int|total_reviews|avg_score|
+----------------+-------------+---------+
|               1|        11424|      1.0|
|               2|         3151|      2.0|
|               3|         8179|      3.0|
|               4|        19141|      4.0|
|               5|        57328|      5.0|
+----------------+-------------+---------+



# Bài 6: Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo danh mục sản phẩm


In [26]:
df_products = print_schema("Products.csv")

df = df_order_items \
    .join(df_orders, "Order_ID", "left") \
    .join(df_products, "Product_ID", "left")


df_2024 = df.filter(
    F.year("Order_Purchase_Timestamp") == 2024
)
df_2024 = df_2024.withColumn(
    "revenue",
    F.col("Price") + F.col("Freight_Value")
)
result = df_2024.groupBy("Product_Category_Name") \
    .agg(
        F.sum("revenue").alias("total_revenue")
    ) \
    .orderBy(F.desc("total_revenue"))

result.show()

Schema of Products.csv
root
 |-- Product_ID: string (nullable = true)
 |-- Product_Category_Name: string (nullable = true)
 |-- Product_Weight_Gr: integer (nullable = true)
 |-- Product_Length_Cm: integer (nullable = true)
 |-- Product_Height_Cm: integer (nullable = true)
 |-- Product_Width_Cm: integer (nullable = true)

+---------------------+------------------+
|Product_Category_Name|     total_revenue|
+---------------------+------------------+
|        Health_Beauty| 885191.1200000007|
|        Watches_Gifts| 771986.7499999991|
|       Bed_Bath_Table| 650794.6999999994|
|       Sports_Leisure| 621999.3399999996|
| Computers_Accesso...| 594771.0400000003|
|           Housewares| 491576.9600000005|
|      Furniture_Decor|476466.12999999983|
|                 Auto|404210.56999999995|
|                 Baby|299052.56000000006|
|           Cool_Stuff|273910.05000000005|
|         Garden_Tools|259068.31999999983|
|            Telephony|217452.12999999963|
|            Perfumery|204562.53

# Bài 7: Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình cho từng sản phẩm


In [30]:
df_reviews = print_schema("Order_Reviews.csv")

df = df_order_items \
    .join(df_products, "Product_ID", "left") \
    .join(df_orders, "Order_ID", "left") \
    .join(df_reviews, "Order_ID", "left")

df = df.withColumn(
    "Review_Score_Int",
    F.col("Review_Score").cast("int")
).filter(
    F.col("Review_Score_Int").isNotNull()
)

product_stats = df.groupBy("Product_ID", "Product_Category_Name") \
    .agg(
        F.count("*").alias("total_sold"),
        F.avg("Review_Score_Int").alias("avg_rating")
    )
top_product = product_stats.orderBy(F.desc("total_sold")).limit(1)

top_product.show()


Schema of Order_Reviews.csv
root
 |-- Review_ID: string (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Review_Score: string (nullable = true)
 |-- Review_Comment_Title_En: string (nullable = true)
 |-- Review_Comment_Message_En: string (nullable = true)
 |-- Review_Creation_Date: string (nullable = true)
 |-- Review_Answer_Timestamp: timestamp (nullable = true)

+--------------------+---------------------+----------+-----------------+
|          Product_ID|Product_Category_Name|total_sold|       avg_rating|
+--------------------+---------------------+----------+-----------------+
|aca2eb7d00ea1a7b8...|      Furniture_Decor|       524|4.019083969465649|
+--------------------+---------------------+----------+-----------------+



In [ ]:
# Top N
product_stats.orderBy(F.desc("total_sold")).show(10)

+--------------------+---------------------+----------+------------------+
|          Product_ID|Product_Category_Name|total_sold|        avg_rating|
+--------------------+---------------------+----------+------------------+
|aca2eb7d00ea1a7b8...|      Furniture_Decor|       524| 4.019083969465649|
|422879e10f4668299...|         Garden_Tools|       486|3.9465020576131686|
|99a4788cb24856965...|       Bed_Bath_Table|       482|3.8983402489626555|
|389d119b48cf3043d...|         Garden_Tools|       391| 4.117647058823529|
|368c6c730842d7801...|         Garden_Tools|       388| 3.922680412371134|
|53759a2ecddad2bb8...|         Garden_Tools|       373| 3.868632707774799|
|d1c427060a0f73f6b...| Computers_Accesso...|       340| 4.194117647058824|
|53b36df67ebb7c415...|        Watches_Gifts|       320|          4.190625|
|154e7e31ebfa09220...|        Health_Beauty|       292| 4.315068493150685|
|3dd2a17168ec895c7...| Computers_Accesso...|       272| 4.209558823529412|
+--------------------+---

# Bài 10: Xếp hạng các seller dựa trên tổng doanh thu và số lượng đơn hàng bán được.


In [36]:
df = df_order_items.withColumn(
    "revenue",
    F.col("Price") + F.col("Freight_Value")
)

seller_stats = df.groupBy("Seller_ID") \
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.countDistinct("Order_ID").alias("total_orders"),
        F.count("*").alias("total_items_sold")
    )

from pyspark.sql.window import Window

w_rev = Window.orderBy(F.desc("total_revenue"))

seller_stats = seller_stats.withColumn(
    "rank_by_revenue",
    F.rank().over(w_rev)
)
w_order = Window.orderBy(F.desc("total_orders"))

seller_stats = seller_stats.withColumn(
    "rank_by_orders",
    F.rank().over(w_order)
)
seller_stats.orderBy("rank_by_revenue").show(20)

26/05/20 14:15:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 14:15:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 14:15:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 14:15:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 14:15:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 14:15:22 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/20 1

In [37]:
spark.stop()